In [ ]:
import os
import pandas as pd
import requests
from bs4 import BeautifulSoup


def get_article_text(url):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 Chrome/124.0 Safari/537.36"
        )
    }

    response = requests.get(url, headers=headers, timeout=15)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    paragraphs = soup.find_all("p")
    article_text = "\n".join(
        p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)
    )

    return article_text


def main():
    df = pd.read_csv("articles.csv")

    os.makedirs("articles", exist_ok=True)

    total = len(df)

    for i, row in df.iterrows():
        title = row["title"]
        url = row["url"]

        print(f"[{i+1}/{total}] Processing: {title}")

        try:
            article_text = get_article_text(url)

            output_file = os.path.join("articles", f"art{i+1}.txt")

            with open(output_file, "w", encoding="utf-8") as f:
                f.write(f"Title: {title}\n")
                f.write(f"URL: {url}\n")
                f.write("\n")
                f.write(article_text)

            print(f"    ✓ Saved to {output_file} ({len(article_text)} characters)")

        except Exception as e:
            print(f"    ✗ Failed: {e}")

    print("\nDone!")


if __name__ == "__main__":
    main()